In [16]:
%pip install -U ipywidgets

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 27.8 MB/s  0:00:00

  Attempting uninstall: widgetsnbextension

    Found existing installation: widgetsnbextension 4.0.15

    Uninstalling widgetsnbextension-4.0.15:

      Successfully uninstalled widgetsnbextension-4.0.15

  Attempting uninstall: jupyterlab_widgets

    Found existing installation: jupyterlab_widgets 3.0.16

    Uninstalling jupyterlab_widgets-3.0.16:

      Successfully uninstalled jupyterlab_widgets-3.0.16

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
  Attempting uninstall: ipywidgets
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
    Found existing installation: ipywidgets 8.1.8
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
    Uninstalling ipywidgets-8.1.8:
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
      Successfully uninstalled ipywidgets-8.1.

In [17]:
print("Customer Churn Intelligence")
print("Python environment is working!")

Customer Churn Intelligence
Python environment is working!


In [18]:
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.5


In [19]:
import duckdb

print("DuckDB is working!")


DuckDB is working!


In [20]:
from pathlib import Path

data_path = Path("../data/raw/hm")

print("Data folder:", data_path)
print("Files found:")

for file in data_path.iterdir():
    print(file.name)

Data folder: ..\data\raw\hm
Files found:
.gitkeep
articles.csv
customers.csv
transactions_train.csv


In [21]:
import duckdb

con = duckdb.connect()

con.sql("""
SELECT COUNT(*) AS total_transactions
FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│           31788324 │
└────────────────────┘



In [22]:
import duckdb

con = duckdb.connect()

In [23]:
con.sql("""
SELECT COUNT(DISTINCT customer_id) AS distinct_customers
FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ distinct_customers │
│       int64        │
├────────────────────┤
│            1362281 │
└────────────────────┘



In [24]:
con.sql("""
SELECT
    MIN(t_dat) AS earliest_date,
    MAX(t_dat) AS latest_date
FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
""").show()

┌───────────────┬─────────────┐
│ earliest_date │ latest_date │
│     date      │    date     │
├───────────────┼─────────────┤
│ 2018-09-20    │ 2020-09-22  │
└───────────────┴─────────────┘



In [25]:
con.sql("""
SELECT COUNT(*) AS total_customers
FROM read_csv_auto('../data/raw/hm/customers.csv')
""").show()

┌─────────────────┐
│ total_customers │
│      int64      │
├─────────────────┤
│         1371980 │
└─────────────────┘



In [26]:
con.sql("""
DESCRIBE
SELECT *
FROM read_csv_auto('../data/raw/hm/customers.csv')
""").show()

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ FN                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Active                 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ club_member_status     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ fashion_news_frequency │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ age                    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ postal_code            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [27]:
con.sql("""
SELECT
    COUNT(*) AS total_customers,
    COUNT(*) FILTER (WHERE age IS NULL) AS missing_age,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE age IS NULL) / COUNT(*),
        2
    ) AS missing_age_percentage
FROM read_csv_auto('../data/raw/hm/customers.csv')
""").show()

┌─────────────────┬─────────────┬────────────────────────┐
│ total_customers │ missing_age │ missing_age_percentage │
│      int64      │    int64    │         double         │
├─────────────────┼─────────────┼────────────────────────┤
│         1371980 │       15861 │                   1.16 │
└─────────────────┴─────────────┴────────────────────────┘



In [28]:
con.sql("""
SELECT
    customer_id,
    t_dat,
    LAG(t_dat) OVER (
        PARTITION BY customer_id
        ORDER BY t_dat
    ) AS previous_purchase
FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
LIMIT 20
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────────────────────────────────┬────────────┬───────────────────┐
│                           customer_id                            │   t_dat    │ previous_purchase │
│                             varchar                              │    date    │       date        │
├──────────────────────────────────────────────────────────────────┼────────────┼───────────────────┤
│ 0010876c57885b66aea0c7787b608ca17725fc6882e270c7c01d3d5b90c7ccba │ 2018-11-01 │ NULL              │
│ 0010876c57885b66aea0c7787b608ca17725fc6882e270c7c01d3d5b90c7ccba │ 2018-11-01 │ 2018-11-01        │
│ 0010876c57885b66aea0c7787b608ca17725fc6882e270c7c01d3d5b90c7ccba │ 2018-11-01 │ 2018-11-01        │
│ 0010876c57885b66aea0c7787b608ca17725fc6882e270c7c01d3d5b90c7ccba │ 2018-11-01 │ 2018-11-01        │
│ 0010876c57885b66aea0c7787b608ca17725fc6882e270c7c01d3d5b90c7ccba │ 2018-11-01 │ 2018-11-01        │
│ 0010876c57885b66aea0c7787b608ca17725fc6882e270c7c01d3d5b90c7ccba │ 2019-05-02 │ 

In [29]:
con.sql("""
WITH unique_purchases AS (
    SELECT DISTINCT
        customer_id,
        t_dat
    FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
),

purchases_with_previous AS (
    SELECT
        customer_id,
        t_dat,
        LAG(t_dat) OVER (
            PARTITION BY customer_id
            ORDER BY t_dat
        ) AS previous_purchase
    FROM unique_purchases
),

gaps AS (
    SELECT
        customer_id,
        date_diff('day', previous_purchase, t_dat) AS gap_days
    FROM purchases_with_previous
    WHERE previous_purchase IS NOT NULL
)

SELECT
    MIN(gap_days) AS minimum_gap,
    quantile_cont(gap_days, 0.25) AS percentile_25,
    quantile_cont(gap_days, 0.50) AS median_gap,
    quantile_cont(gap_days, 0.75) AS percentile_75,
    quantile_cont(gap_days, 0.90) AS percentile_90,
    MAX(gap_days) AS maximum_gap
FROM gaps
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬───────────────┬────────────┬───────────────┬───────────────┬─────────────┐
│ minimum_gap │ percentile_25 │ median_gap │ percentile_75 │ percentile_90 │ maximum_gap │
│    int64    │    double     │   double   │    double     │    double     │    int64    │
├─────────────┼───────────────┼────────────┼───────────────┼───────────────┼─────────────┤
│           1 │           7.0 │       22.0 │          58.0 │         124.0 │         731 │
└─────────────┴───────────────┴────────────┴───────────────┴───────────────┴─────────────┘



In [30]:
import pandas as pd
import numpy as np
import duckdb
import sklearn

print("Everything is working!")

Everything is working!


In [31]:
import duckdb, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

con = duckdb.connect()

con.sql("""
CREATE OR REPLACE TABLE customer_features AS
WITH tx AS (
    SELECT customer_id, CAST(t_dat AS DATE) AS t_dat, article_id, price, sales_channel_id
    FROM read_csv_auto('data/raw/hm/transactions_train.csv')
),
feature_window AS (SELECT * FROM tx WHERE t_dat <= DATE '2020-05-25'),
label_window AS (
    SELECT DISTINCT customer_id FROM tx
    WHERE t_dat > DATE '2020-05-25' AND t_dat <= DATE '2020-09-22'
),
agg AS (
    SELECT customer_id,
        MIN(t_dat) AS first_purchase, MAX(t_dat) AS last_purchase,
        COUNT(*) AS total_orders, SUM(price) AS total_spend, AVG(price) AS avg_order_value,
        COUNT(*) FILTER (WHERE t_dat > DATE '2020-02-25') AS recent_90d_orders,
        COUNT(*) FILTER (WHERE t_dat > DATE '2019-11-27' AND t_dat <= DATE '2020-02-25') AS prior_90d_orders,
        MODE(sales_channel_id) AS preferred_channel
    FROM feature_window GROUP BY customer_id
)
SELECT agg.*,
    DATE_DIFF('day', agg.first_purchase, DATE '2020-05-25') AS tenure_days,
    DATE_DIFF('day', agg.last_purchase, DATE '2020-05-25') AS days_since_last_purchase,
    CASE WHEN lw.customer_id IS NOT NULL THEN 0 ELSE 1 END AS churn
FROM agg LEFT JOIN label_window lw ON agg.customer_id = lw.customer_id
""")

con.sql("""
CREATE OR REPLACE TABLE category_diversity AS
SELECT t.customer_id, COUNT(DISTINCT a.product_group_name) AS category_diversity
FROM read_csv_auto('data/raw/hm/transactions_train.csv') t
JOIN read_csv_auto('data/raw/hm/articles.csv') a ON t.article_id = a.article_id
WHERE CAST(t.t_dat AS DATE) <= DATE '2020-05-25'
GROUP BY t.customer_id
""")

features = con.sql("""
    SELECT f.*, c.category_diversity FROM customer_features f
    LEFT JOIN category_diversity c ON f.customer_id = c.customer_id
""").df()

#customer information and handles missing values missing ages are replaced with the median age

customers = pd.read_csv('data/raw/hm/customers.csv')
customers['FN'] = customers['FN'].fillna(0)
customers['Active'] = customers['Active'].fillna(0)
customers['age'] = customers['age'].fillna(customers['age'].median())
customers['club_member_status'] = customers['club_member_status'].fillna('UNKNOWN')
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].fillna('NONE')

data = features.merge(customers, on='customer_id', how='left')

#Create the 100,000-customer sample

sample, _ = train_test_split(data, train_size=100_000, stratify=data['churn'], random_state=42)
sample = sample.reset_index(drop=True)

def calc_usage_change(row):
    prior, recent = row['prior_90d_orders'], row['recent_90d_orders']
    if prior == 0 and recent == 0: return 0.0
    elif prior == 0: return 200.0
    return ((recent - prior) / prior) * 100
sample['usage_change_pct'] = sample.apply(calc_usage_change, axis=1)

np.random.seed(42)
n = len(sample)
satisfaction = (7.5
    - (sample['usage_change_pct'].clip(upper=0).abs() / 100) * 2
    - (sample['days_since_last_purchase'] / 120).clip(upper=2)
    + np.random.normal(0, 1.0, n))
sample['customer_satisfaction_score'] = satisfaction.clip(1, 10).round(1)

complaint_rate = np.clip((10 - sample['customer_satisfaction_score']) / 10 * 1.5, 0.05, None)
sample['complaints_last_60_days'] = np.random.poisson(complaint_rate)
sample['support_tickets'] = sample['complaints_last_60_days'] + np.random.poisson(0.3, n)
payment_fail_prob = np.clip((10 - sample['customer_satisfaction_score']) / 10 * 0.15, 0.01, 0.3)
sample['payment_failures'] = np.random.binomial(1, payment_fail_prob)

import os
os.makedirs('data/processed', exist_ok=True)
sample.to_parquet('data/processed/modeling_table.parquet', index=False)
print(sample.shape, sample['churn'].mean())

IOException: IO Error: No files found that match the pattern "data/raw/hm/transactions_train.csv"

LINE 5:     FROM read_csv_auto('data/raw/hm/transactions_train.csv')
                 ^

In [ ]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current folder:
c:\Users\PPegu\Omniverse\1 Study\MBA(BE)'27\Projects\Projects\Customer Curn intelligence\Notebooks

Files/folders here:
['01_data_understanding.ipynb']


In [ ]:
import os

print(os.listdir("Data"))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Data'

In [ ]:
import os

print(os.getcwd())

c:\Users\PPegu\Omniverse\1 Study\MBA(BE)'27\Projects\Projects\Customer Curn intelligence\Notebooks


In [ ]:
import os

print(os.listdir("."))

['01_data_understanding.ipynb']


In [ ]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.lower().endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
import os

print(os.path.exists("../data/raw/hm/transactions_train.csv"))

True


In [ ]:
import os

print("Transactions:", os.path.exists("../data/raw/hm/transactions_train.csv"))
print("Customers:", os.path.exists("../data/raw/hm/customers.csv"))
print("Articles:", os.path.exists("../data/raw/hm/articles.csv"))

Transactions: True
Customers: True
Articles: True


In [32]:
import duckdb, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

con = duckdb.connect()

#Customer level features and churn label creation

con.sql("""
CREATE OR REPLACE TABLE customer_features AS
WITH tx AS (
    SELECT customer_id, CAST(t_dat AS DATE) AS t_dat, article_id, price, sales_channel_id
    FROM read_csv_auto('../data/raw/hm/transactions_train.csv')
),
feature_window AS (SELECT * FROM tx WHERE t_dat <= DATE '2020-05-25'),
label_window AS (
    SELECT DISTINCT customer_id FROM tx
    WHERE t_dat > DATE '2020-05-25' AND t_dat <= DATE '2020-09-22'
),
agg AS (
    SELECT customer_id,
        MIN(t_dat) AS first_purchase, MAX(t_dat) AS last_purchase,
        COUNT(*) AS total_orders, SUM(price) AS total_spend, AVG(price) AS avg_order_value,
        COUNT(*) FILTER (WHERE t_dat > DATE '2020-02-25') AS recent_90d_orders,
        COUNT(*) FILTER (WHERE t_dat > DATE '2019-11-27' AND t_dat <= DATE '2020-02-25') AS prior_90d_orders,
        MODE(sales_channel_id) AS preferred_channel
    FROM feature_window GROUP BY customer_id
)
SELECT agg.*,
    DATE_DIFF('day', agg.first_purchase, DATE '2020-05-25') AS tenure_days,
    DATE_DIFF('day', agg.last_purchase, DATE '2020-05-25') AS days_since_last_purchase,
    CASE WHEN lw.customer_id IS NOT NULL THEN 0 ELSE 1 END AS churn
FROM agg LEFT JOIN label_window lw ON agg.customer_id = lw.customer_id
""")

con.sql("""
CREATE OR REPLACE TABLE category_diversity AS
SELECT t.customer_id, COUNT(DISTINCT a.product_group_name) AS category_diversity
FROM read_csv_auto('../data/raw/hm/transactions_train.csv') t
JOIN read_csv_auto('../data/raw/hm/articles.csv') a ON t.article_id = a.article_id
WHERE CAST(t.t_dat AS DATE) <= DATE '2020-05-25'
GROUP BY t.customer_id
""")

features = con.sql("""
    SELECT f.*, c.category_diversity FROM customer_features f
    LEFT JOIN category_diversity c ON f.customer_id = c.customer_id
""").df()

#customer information and handles missing values missing ages are replaced with the median age

customers = pd.read_csv('../data/raw/hm/customers.csv')
customers['FN'] = customers['FN'].fillna(0)
customers['Active'] = customers['Active'].fillna(0)
customers['age'] = customers['age'].fillna(customers['age'].median())
customers['club_member_status'] = customers['club_member_status'].fillna('UNKNOWN')
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].fillna('NONE')

data = features.merge(customers, on='customer_id', how='left')

#Create the 100,000-customer sample

sample, _ = train_test_split(data, train_size=100_000, stratify=data['churn'], random_state=42)
sample = sample.reset_index(drop=True)

def calc_usage_change(row):
    prior, recent = row['prior_90d_orders'], row['recent_90d_orders']
    if prior == 0 and recent == 0: return 0.0
    elif prior == 0: return 200.0
    return ((recent - prior) / prior) * 100
sample['usage_change_pct'] = sample.apply(calc_usage_change, axis=1)

np.random.seed(42)
n = len(sample)
satisfaction = (7.5
    - (sample['usage_change_pct'].clip(upper=0).abs() / 100) * 2
    - (sample['days_since_last_purchase'] / 120).clip(upper=2)
    + np.random.normal(0, 1.0, n))
sample['customer_satisfaction_score'] = satisfaction.clip(1, 10).round(1)

complaint_rate = np.clip((10 - sample['customer_satisfaction_score']) / 10 * 1.5, 0.05, None)
sample['complaints_last_60_days'] = np.random.poisson(complaint_rate)
sample['support_tickets'] = sample['complaints_last_60_days'] + np.random.poisson(0.3, n)
payment_fail_prob = np.clip((10 - sample['customer_satisfaction_score']) / 10 * 0.15, 0.01, 0.3)
sample['payment_failures'] = np.random.binomial(1, payment_fail_prob)

import os
os.makedirs('data/processed', exist_ok=True)
sample.to_parquet('data/processed/modeling_table.parquet', index=False)
print(sample.shape, sample['churn'].mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(100000, 24) 0.59199
